In [ ]:
# ============================================================
# PROJECT 6 — HMM-BASED PROBABILISTIC RETURN FORECASTING
# NIFTY 50
# ============================================================

!pip -q install yfinance hmmlearn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, brier_score_loss

# ------------------------------------------------------------
# 1. DOWNLOAD NIFTY 50 DATA
# ------------------------------------------------------------
df = yf.download(
    "^NSEI",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.dropna()

# ------------------------------------------------------------
# 2. CREATE FEATURES
# ------------------------------------------------------------
df["Return"] = df["Close"].pct_change()

df["Volatility"] = (
    df["Return"]
    .rolling(20)
    .std()
)

df["Momentum"] = (
    df["Close"] /
    df["Close"].rolling(20).mean() - 1
)

df = df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

features = [
    "Return",
    "Volatility",
    "Momentum"
]

# ------------------------------------------------------------
# 3. STANDARDIZE FEATURES
# ------------------------------------------------------------
scaler = StandardScaler()

X = scaler.fit_transform(
    df[features]
)

# ------------------------------------------------------------
# 4. FIT GAUSSIAN HMM
# ------------------------------------------------------------
n_states = 3

model = GaussianHMM(
    n_components=n_states,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

model.fit(X)

df["State"] = model.predict(X)

# ------------------------------------------------------------
# 5. LABEL STATES
# ------------------------------------------------------------
state_return = (
    df.groupby("State")["Return"]
    .mean()
)

ordered_states = (
    state_return
    .sort_values()
    .index
    .tolist()
)

labels = [
    "Bear",
    "Neutral",
    "Bull"
]

state_mapping = {
    ordered_states[i]: labels[i]
    for i in range(n_states)
}

df["Regime"] = (
    df["State"]
    .map(state_mapping)
)

# ------------------------------------------------------------
# 6. REGIME-SPECIFIC RETURN DISTRIBUTIONS
# ------------------------------------------------------------
regime_stats = []

for state in range(n_states):

    returns = (
        df[df["State"] == state]["Return"]
    )

    regime_stats.append({
        "State": state,
        "Regime": state_mapping[state],
        "Mean Return": returns.mean(),
        "Return Volatility": returns.std(),
        "Positive Return Probability":
            (returns > 0).mean()
    })

regime_stats = pd.DataFrame(
    regime_stats
)

print("=" * 70)
print("HMM-BASED PROBABILISTIC RETURN FORECASTING")
print("=" * 70)

print("\nREGIME RETURN CHARACTERISTICS")
print("-" * 70)

display(
    regime_stats
    .assign(
        **{
            "Mean Return (%)":
                regime_stats["Mean Return"] * 100,
            "Volatility (%)":
                regime_stats["Return Volatility"] * 100,
            "Positive Return Probability (%)":
                regime_stats[
                    "Positive Return Probability"
                ] * 100
        }
    )[[
        "State",
        "Regime",
        "Mean Return (%)",
        "Volatility (%)",
        "Positive Return Probability (%)"
    ]].round(4)
)

# ------------------------------------------------------------
# 7. ONE-STEP FUTURE REGIME PROBABILITIES
# ------------------------------------------------------------
transition = model.transmat_

state_probs = model.predict_proba(X)

future_state_probs = (
    state_probs @ transition
)

# ------------------------------------------------------------
# 8. EXPECTED FUTURE RETURN
# ------------------------------------------------------------
state_mean_returns = np.zeros(n_states)

for state in range(n_states):

    state_mean_returns[state] = (
        df[df["State"] == state]["Return"]
        .mean()
    )

df["Expected_Return_1D"] = (
    future_state_probs @
    state_mean_returns
)

# Probability of positive return
positive_probability = np.zeros(n_states)

for state in range(n_states):

    positive_probability[state] = (
        df[df["State"] == state]["Return"] > 0
    ).mean()

df["Probability_Positive_Return"] = (
    future_state_probs @
    positive_probability
)

# ------------------------------------------------------------
# 9. PROBABILISTIC FORECAST
# ------------------------------------------------------------
print("\nCURRENT FORECAST")
print("-" * 70)

latest_forecast = df.iloc[-1]

print(
    "Date:",
    df.index[-1].date()
)

print(
    "Current Regime:",
    latest_forecast["Regime"]
)

print(
    f"\nExpected Next-Day Return: "
    f"{latest_forecast['Expected_Return_1D'] * 100:.4f}%"
)

print(
    f"Probability of Positive Return: "
    f"{latest_forecast['Probability_Positive_Return'] * 100:.2f}%"
)

print("\nNext-Day Regime Probabilities:")

for state in range(n_states):

    print(
        f"{state_mapping[state]}: "
        f"{future_state_probs[-1, state] * 100:.2f}%"
    )

# ------------------------------------------------------------
# 10. MULTI-HORIZON REGIME PROBABILITIES
# ------------------------------------------------------------
print("\nMULTI-HORIZON REGIME FORECAST")
print("-" * 70)

current_probs = state_probs[-1]

horizons = [
    1,
    5,
    10,
    20
]

forecast_rows = []

for horizon in horizons:

    probabilities = (
        current_probs @
        np.linalg.matrix_power(
            transition,
            horizon
        )
    )

    expected_return = (
        probabilities @
        state_mean_returns
    )

    probability_positive = (
        probabilities @
        positive_probability
    )

    forecast_rows.append({
        "Horizon (Days)": horizon,
        "Bear Probability (%)":
            probabilities[ordered_states.index(
                ordered_states[0]
            )] * 100,
        "Expected Return (%)":
            expected_return * 100,
        "Positive Return Probability (%)":
            probability_positive * 100
    })

forecast_df = pd.DataFrame(
    forecast_rows
)

print(
    forecast_df.round(4).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 11. OUT-OF-SAMPLE ONE-DAY FORECAST
# ------------------------------------------------------------
# Compare predicted probability with actual next-day direction.

actual_positive = (
    df["Return"].shift(-1) > 0
).astype(int)

forecast_probability = (
    df["Probability_Positive_Return"]
)

evaluation = pd.DataFrame({
    "Probability": forecast_probability,
    "Actual": actual_positive
}).dropna()

# Remove any probabilities outside numerical bounds
evaluation["Probability"] = np.clip(
    evaluation["Probability"],
    1e-6,
    1 - 1e-6
)

# Brier Score
brier = brier_score_loss(
    evaluation["Actual"],
    evaluation["Probability"]
)

# Log Loss
logloss = log_loss(
    evaluation["Actual"],
    evaluation["Probability"]
)

# Directional accuracy
direction_accuracy = (
    (
        evaluation["Probability"] >= 0.5
    ).astype(int)
    ==
    evaluation["Actual"]
).mean()

print("\nPROBABILISTIC FORECAST EVALUATION")
print("-" * 70)

print(
    f"Brier Score       : {brier:.4f}"
)

print(
    f"Log Loss          : {logloss:.4f}"
)

print(
    f"Directional Accuracy: "
    f"{direction_accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# 12. FORECAST CALIBRATION TABLE
# ------------------------------------------------------------
evaluation["Probability_Bin"] = pd.cut(
    evaluation["Probability"],
    bins=[
        0,
        0.2,
        0.4,
        0.6,
        0.8,
        1.0
    ]
)

calibration = (
    evaluation
    .groupby(
        "Probability_Bin",
        observed=False
    )
    .agg(
        Forecast_Probability=("Probability", "mean"),
        Actual_Positive_Rate=("Actual", "mean"),
        Observations=("Actual", "count")
    )
)

print("\nFORECAST CALIBRATION")
print("-" * 70)

print(
    calibration.round(4).to_string()
)

# ------------------------------------------------------------
# 13. FORECAST PROBABILITY THROUGH TIME
# ------------------------------------------------------------
plt.figure(
    figsize=(15, 6)
)

plt.plot(
    df.index,
    df["Probability_Positive_Return"] * 100,
    linewidth=0.8
)

plt.axhline(
    50,
    linestyle="--",
    linewidth=1
)

plt.title(
    "HMM Probability of Positive Next-Day Return"
)

plt.xlabel("Date")
plt.ylabel(
    "Probability of Positive Return (%)"
)

plt.grid(alpha=0.25)

plt.show()

# ------------------------------------------------------------
# 14. EXPECTED RETURN THROUGH TIME
# ------------------------------------------------------------
plt.figure(
    figsize=(15, 6)
)

plt.plot(
    df.index,
    df["Expected_Return_1D"] * 100,
    linewidth=0.8
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.title(
    "HMM Expected Next-Day Return"
)

plt.xlabel("Date")
plt.ylabel(
    "Expected Return (%)"
)

plt.grid(alpha=0.25)

plt.show()

# ------------------------------------------------------------
# 15. SAVE RESULTS
# ------------------------------------------------------------
df.to_csv(
    "NIFTY50_HMM_Probabilistic_Return_Forecast.csv"
)

forecast_df.to_csv(
    "HMM_Multi_Horizon_Forecast.csv",
    index=False
)

calibration.to_csv(
    "HMM_Forecast_Calibration.csv"
)

print("\nFiles saved:")
print(
    "• NIFTY50_HMM_Probabilistic_Return_Forecast.csv"
)
print(
    "• HMM_Multi_Horizon_Forecast.csv"
)
print(
    "• HMM_Forecast_Calibration.csv"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 3.5 MB/s eta 0:00:00
